In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV, StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

import matplotlib.pyplot as plt
import seaborn as sns

import pickle

In [2]:
df = pd.read_csv("Data/train.csv", index_col='PassengerId')

In [3]:
y = df.Survived
df = df.drop(columns=['Survived'])

### Cabin feature

In [4]:
df['HasCabin'] = df.Cabin.notna().astype(int)

In [5]:
df['Deck'] = df['Cabin'].str[0].fillna('U')

In [6]:
df['CabinCount'] = (df['Cabin'].str.count(' ')+1).fillna(0).astype('int')

In [7]:
df = df.drop(columns=['Cabin'])

In [8]:
df.head()

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,HasCabin,Deck,CabinCount
PassengerId,,,,,,,,,,,,
1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,0,U,0
2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,1,C,1
3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,0,U,0
4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,1,C,1
5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,0,U,0


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 891 entries, 1 to 891
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Pclass      891 non-null    int64  
 1   Name        891 non-null    object 
 2   Sex         891 non-null    object 
 3   Age         714 non-null    float64
 4   SibSp       891 non-null    int64  
 5   Parch       891 non-null    int64  
 6   Ticket      891 non-null    object 
 7   Fare        891 non-null    float64
 8   Embarked    889 non-null    object 
 9   HasCabin    891 non-null    int64  
 10  Deck        891 non-null    object 
 11  CabinCount  891 non-null    int64  
dtypes: float64(2), int64(5), object(5)
memory usage: 90.5+ KB


### Ticket feature

In [10]:
ticket_counts = df.Ticket.value_counts()
df['GroupSize'] = df.Ticket.map(ticket_counts)

In [11]:
df['TicketPrefix'] = (df['Ticket']
                        .str.split()
                        .apply(lambda x: x[0].replace('.', '').replace('/', '') 
                        if len(x)>1 else 'NoPrefix'))

In [12]:
df = df.drop(columns=['Ticket'])

In [13]:
treshhold = 0.005 
prefix_freq = df['TicketPrefix'].value_counts(normalize=True)
rare_prefixes = prefix_freq[prefix_freq < treshhold].index
rare_prefixes

Index(['SCParis', 'SOPP', 'SCAH', 'WEP', 'PP', 'PPP', 'SOTONO2', 'SWPP', 'SP',
       'SCA4', 'SCOW', 'SOP', 'Fa', 'AS', 'SC', 'FC', 'CASOTON'],
      dtype='object', name='TicketPrefix')

In [14]:
df['TicketPrefix'] = df['TicketPrefix'].replace(rare_prefixes, 'Rare')

In [15]:
df['TicketPrefix'].value_counts()

TicketPrefix
NoPrefix    665
PC           60
CA           41
Rare         31
A5           21
SOTONOQ      15
STONO        12
WC           10
SCPARIS       7
A4            7
STONO2        6
SOC           6
C             5
FCC           5
Name: count, dtype: int64

In [16]:
df.head()

,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,HasCabin,Deck,CabinCount,GroupSize,TicketPrefix
PassengerId,,,,,,,,,,,,,
1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S,0,U,0,1,A5
2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C,1,C,1,1,PC
3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S,0,U,0,1,STONO2
4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S,1,C,1,2,NoPrefix
5,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S,0,U,0,1,NoPrefix


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 891 entries, 1 to 891
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Pclass        891 non-null    int64  
 1   Name          891 non-null    object 
 2   Sex           891 non-null    object 
 3   Age           714 non-null    float64
 4   SibSp         891 non-null    int64  
 5   Parch         891 non-null    int64  
 6   Fare          891 non-null    float64
 7   Embarked      889 non-null    object 
 8   HasCabin      891 non-null    int64  
 9   Deck          891 non-null    object 
 10  CabinCount    891 non-null    int64  
 11  GroupSize     891 non-null    int64  
 12  TicketPrefix  891 non-null    object 
dtypes: float64(2), int64(6), object(5)
memory usage: 97.5+ KB


### Name feature

In [18]:
df['NameLen'] = df['Name'].apply(len)

In [19]:
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

#### Title

In [20]:
title_groups = {
    'Mr': ['Mr'],
    'Miss': ['Miss', 'Mlle', 'Ms'],
    'Mrs': ['Mrs', 'Mme'],
    'Master': ['Master'],
    'Rare': ['Dr', 'Rev', 'Col', 'Major', 'Capt', 'Sir', 'Lady', 'Don', 'Countess',
              'Jonkheer', 'Dona']
}

flat_mapping = {}

for target_tit, source_tit_list in title_groups.items():
    for source_title in source_tit_list:
        flat_mapping[source_title] = target_tit


In [21]:
df['Title'] = df['Title'].map(flat_mapping).fillna('Rare')

In [22]:
df = df.drop(columns=['Name'])

In [23]:
display(df.head())
df.info()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,HasCabin,Deck,CabinCount,GroupSize,TicketPrefix,NameLen,Title
PassengerId,,,,,,,,,,,,,,
1,3,male,22.0,1,0,7.2500,S,0,U,0,1,A5,23,Mr
2,1,female,38.0,1,0,71.2833,C,1,C,1,1,PC,51,Mrs
3,3,female,26.0,0,0,7.9250,S,0,U,0,1,STONO2,22,Miss
4,1,female,35.0,1,0,53.1000,S,1,C,1,2,NoPrefix,44,Mrs
5,3,male,35.0,0,0,8.0500,S,0,U,0,1,NoPrefix,24,Mr


<class 'pandas.core.frame.DataFrame'>
Index: 891 entries, 1 to 891
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Pclass        891 non-null    int64  
 1   Sex           891 non-null    object 
 2   Age           714 non-null    float64
 3   SibSp         891 non-null    int64  
 4   Parch         891 non-null    int64  
 5   Fare          891 non-null    float64
 6   Embarked      889 non-null    object 
 7   HasCabin      891 non-null    int64  
 8   Deck          891 non-null    object 
 9   CabinCount    891 non-null    int64  
 10  GroupSize     891 non-null    int64  
 11  TicketPrefix  891 non-null    object 
 12  NameLen       891 non-null    int64  
 13  Title         891 non-null    object 
dtypes: float64(2), int64(7), object(5)
memory usage: 104.4+ KB


In [24]:
df[df.Embarked.isna()]

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,HasCabin,Deck,CabinCount,GroupSize,TicketPrefix,NameLen,Title
PassengerId,,,,,,,,,,,,,,
62,1,female,38.0,0,0,80.0,NaN,1,B,1,2,NoPrefix,19,Miss
830,1,female,62.0,0,0,80.0,NaN,1,B,1,2,NoPrefix,41,Mrs


In [25]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

df_clust = df.copy()

df_clust['Age'] = df_clust['Age'].fillna(df_clust['Age'].median())

features_to_use = [col for col in df_clust.columns if col not in ['PassengerId', 'Survived', 'Embarked']]

df_encoded = pd.get_dummies(
    df_clust[features_to_use], 
    columns=['Sex', 'Deck', 'TicketPrefix', 'Title'], 
    drop_first=True
)

scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_encoded)

dbscan = DBSCAN(eps=3.5, min_samples=4)
df_clust['Cluster'] = dbscan.fit_predict(df_scaled)

nan_indices = df[df['Embarked'].isna()].index
nan_clusters = df_clust.loc[nan_indices, 'Cluster'].values

print(f"Clasters with our NaN embarked passangers: {nan_clusters}")

for cluster in set(nan_clusters):
    if cluster == -1:
        print("\nNoise")
    else:
        # Берем всех людей из этого кластера, у которых Embarked не NaN
        cluster_members = df_clust[(df_clust['Cluster'] == cluster) & (df_clust['Embarked'].notna())]
        print(f"\nPorts' distribution into cluster №{cluster} (total similar passangers: {len(cluster_members)}):")
        print(cluster_members['Embarked'].value_counts())

Clasters with our NaN embarked passangers: [15 30]

Ports' distribution into cluster №30 (total similar passangers: 6):
Embarked
S    4
C    2
Name: count, dtype: int64

Ports' distribution into cluster №15 (total similar passangers: 10):
Embarked
S    8
C    2
Name: count, dtype: int64


In [26]:
df['Embarked'] = df['Embarked'].fillna('S')

In [27]:
display(df.head())
df.info()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,HasCabin,Deck,CabinCount,GroupSize,TicketPrefix,NameLen,Title
PassengerId,,,,,,,,,,,,,,
1,3,male,22.0,1,0,7.2500,S,0,U,0,1,A5,23,Mr
2,1,female,38.0,1,0,71.2833,C,1,C,1,1,PC,51,Mrs
3,3,female,26.0,0,0,7.9250,S,0,U,0,1,STONO2,22,Miss
4,1,female,35.0,1,0,53.1000,S,1,C,1,2,NoPrefix,44,Mrs
5,3,male,35.0,0,0,8.0500,S,0,U,0,1,NoPrefix,24,Mr


<class 'pandas.core.frame.DataFrame'>
Index: 891 entries, 1 to 891
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Pclass        891 non-null    int64  
 1   Sex           891 non-null    object 
 2   Age           714 non-null    float64
 3   SibSp         891 non-null    int64  
 4   Parch         891 non-null    int64  
 5   Fare          891 non-null    float64
 6   Embarked      891 non-null    object 
 7   HasCabin      891 non-null    int64  
 8   Deck          891 non-null    object 
 9   CabinCount    891 non-null    int64  
 10  GroupSize     891 non-null    int64  
 11  TicketPrefix  891 non-null    object 
 12  NameLen       891 non-null    int64  
 13  Title         891 non-null    object 
dtypes: float64(2), int64(7), object(5)
memory usage: 136.7+ KB


### Categorical type casting

In [28]:
object_columns = df.select_dtypes(['object']).columns
df[object_columns] = df[object_columns].astype('category')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 891 entries, 1 to 891
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   Pclass        891 non-null    int64   
 1   Sex           891 non-null    category
 2   Age           714 non-null    float64 
 3   SibSp         891 non-null    int64   
 4   Parch         891 non-null    int64   
 5   Fare          891 non-null    float64 
 6   Embarked      891 non-null    category
 7   HasCabin      891 non-null    int64   
 8   Deck          891 non-null    category
 9   CabinCount    891 non-null    int64   
 10  GroupSize     891 non-null    int64   
 11  TicketPrefix  891 non-null    category
 12  NameLen       891 non-null    int64   
 13  Title         891 non-null    category
dtypes: category(5), float64(2), int64(7)
memory usage: 107.7 KB


### One-Hot and KNNimputer

In [30]:
X = df

In [31]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 891 entries, 1 to 891
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   Pclass        891 non-null    int64   
 1   Sex           891 non-null    category
 2   Age           714 non-null    float64 
 3   SibSp         891 non-null    int64   
 4   Parch         891 non-null    int64   
 5   Fare          891 non-null    float64 
 6   Embarked      891 non-null    category
 7   HasCabin      891 non-null    int64   
 8   Deck          891 non-null    category
 9   CabinCount    891 non-null    int64   
 10  GroupSize     891 non-null    int64   
 11  TicketPrefix  891 non-null    category
 12  NameLen       891 non-null    int64   
 13  Title         891 non-null    category
dtypes: category(5), float64(2), int64(7)
memory usage: 107.7 KB
